# AquaTutor

Local YOLO student–teacher PoC



In [9]:
# Imports
from copy import deepcopy
from pathlib import Path
from random import Random
from shutil import copy2
from tempfile import TemporaryDirectory

import numpy as np
import torch
import yaml
from PIL import Image
from ultralytics import YOLO
from ultralytics.cfg import get_cfg


# Data Path Constants

CHECKPOINT = Path(r"C:\SmartBay\Analysis\experiments\Brackish_200\weights\best.pt")

ROOT = Path(r"C:\SmartBay\DataSets\Brackish")

TRAIN_IMAGES = ROOT / "train" / "images"
TRAIN_LABELS = ROOT / "train" / "labels"

EVAL_IMAGES = ROOT / "valid" / "images"
EVAL_LABELS = ROOT / "valid" / "labels"

# Params
X = 8                 # Images sampled per split
SEED = 42
DEVICE = "cpu"        # Change to "cuda:0" for a local NVIDIA GPU.
IMAGE_SIZE = 320
BRIGHTNESS = 1.2      # Brighten training images by 20% before clipping.
DECAY = 0.90

# Create a small sample set to run locally

Pick x Samples and Lables

In [10]:
print("Training folder:", TRAIN_IMAGES)
print("Folder exists:", TRAIN_IMAGES.is_dir())

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
images = [
    p for p in TRAIN_IMAGES.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
]
print("Images found:", len(images))
print("Requested:", X)

Training folder: C:\SmartBay\DataSets\Brackish\train\images
Folder exists: True
Images found: 11738
Requested: 8


In [11]:
# Sample images with matching YOLO label files

rng = Random(SEED)
torch.manual_seed(SEED)

# build pairs
def sample_pairs(images_dir, labels_dir):

    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    pairs = []

    for image in sorted(images_dir.rglob("*")):
        if image.suffix.lower() in extensions:
            label = labels_dir / image.relative_to(images_dir).with_suffix(".txt")
            pairs.append((image, label))

    return rng.sample(pairs, X)

# splits
train_pairs = sample_pairs(TRAIN_IMAGES, TRAIN_LABELS)
eval_pairs = sample_pairs(EVAL_IMAGES, EVAL_LABELS)

# print splits data
for split, pairs in (("Train", train_pairs), ("Validation", eval_pairs)):
    print(split, len(pairs), [image.name for image, _ in pairs])

Train 8 ['2019-03-21_03-21-54to2019-03-21_03-22-09_1-0071_jpg.rf.7f4354ea756e5cdeb9c08817c3680fd2.jpg', '2019-02-21_06-56-08to2019-02-21_06-56-28_1-0190_jpg.rf.ea81224ce6d5d484ccf3732482c42ca3.jpg', '2019-02-20_19-19-23to2019-02-20_19-19-40_1-0191_jpg.rf.150d255994ded977f4e36d01e64ba257.jpg', '2019-03-06_22-04-39to2019-03-06_22-04-49_1-0134_jpg.rf.c4c383f35b7df490e3c01186118cc66d.jpg', '2019-03-06_22-03-36to2019-03-06_22-04-08_1-0007_jpg.rf.beb5950543c1d6f5e421fac6fbfe276c.jpg', '2019-02-26_00-44-33to2019-02-26_00-44-47_1-0148_jpg.rf.ba8ecb56a8670b74648f031c9a4e7de6.jpg', '2019-02-22_22-21-06to2019-02-22_22-21-16_1-0029_jpg.rf.d30b7970a71a59aff920142aa7e2268e.jpg', '2019-02-21_06-56-08to2019-02-21_06-56-28_1-0017_jpg.rf.5f608f24559500b49fafbbe3b3158899.jpg']
Validation 8 ['2019-03-21_04-10-54to2019-03-21_04-11-06_1-0069_jpg.rf.dc3899721dabec5e318952985429f209.jpg', '2019-03-20_20-08-39to2019-03-20_20-08-48_1-0036_jpg.rf.80547daade80ffcddf8481c7bbfedc4b.jpg', '2019-02-21_06-50-54to2019-

In [12]:
# Create student and teacer models from the same base
student = YOLO(str(CHECKPOINT))
teacher = deepcopy(student)

# teacher swith off GD
teacher.model.to(DEVICE).float().requires_grad_(False).eval()

# access student model weights 
network = student.model
network.to(DEVICE)
network.float()
network.requires_grad_(True)
network.train()
network.args = get_cfg()

images, boxes, batch_indices = [], [], []

for index, (image_path, label_path) in enumerate(train_pairs):

    with Image.open(image_path) as image:
        pixels = np.array(image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE)))

    images.append(torch.from_numpy(pixels).permute(2, 0, 1))

    for line in label_path.read_text(encoding="utf-8-sig").splitlines():
        if not line.strip():
            continue
        boxes.append([float(value) for value in line.split()])
        batch_indices.append(index)

# agument images Brightness changes pixels only. The bounding boxes keep their coordinates.
training_images = torch.stack(images).to(DEVICE, dtype=torch.float32) / 255
training_images = (training_images * BRIGHTNESS).clamp(0, 1)

targets = torch.tensor(boxes, dtype=torch.float32, device=DEVICE)

batch = {
    "img": training_images,
    "batch_idx": torch.tensor(batch_indices, device=DEVICE),
    "cls": targets[:, :1],
    "bboxes": targets[:, 1:],
}

optimizer = torch.optim.SGD(network.parameters(), lr=0.001)
optimizer.zero_grad(set_to_none=True)

loss, _ = network(batch)
loss = loss.sum()
loss.backward()

optimizer.step()

print(f"Student training loss: {loss.item():.4f} on {len(images)} images")

Student training loss: 36.9270 on 8 images


# Update the teacher

Use **90% of the old teacher weight + 10% of the student's weight after SGD**. 

This is the exponential moving average (EMA). 

The table shows the eight weights that changed most in the first convolution. The teacher's starting weights also give us the student's values before SGD because both models started from the same checkpoint. 

BatchNorm running statistics stay at the teacher's starting values for this one-step example.

In [13]:
student_params = dict(network.named_parameters())
teacher_params = dict(teacher.model.named_parameters())
layer = next(name for name, weight in student_params.items() if weight.ndim == 4)

with torch.no_grad():
    student_weights = student_params[layer].flatten()
    teacher_weights = teacher_params[layer].flatten()
    changes = (student_weights - teacher_weights).abs()
    indices = changes.topk(8).indices
    before_sgd = teacher_weights[indices]
    after_sgd = student_weights[indices]

    for name, teacher_weight in teacher_params.items():
        teacher_weight.lerp_(student_params[name], 1 - DECAY)

    teacher_after = teacher_params[layer].flatten()[indices]

expected = DECAY * before_sgd + (1 - DECAY) * after_sgd
torch.testing.assert_close(teacher_after, expected)

print(layer)
print(f"{'Index':>6} {'Both before SGD':>16} {'Student after SGD':>18} {'Teacher after EMA':>18}")
for i, before, student_value, teacher_value in zip(
    indices.tolist(), before_sgd.tolist(), after_sgd.tolist(), teacher_after.tolist()
):
    print(f"{i:6d} {before:16.8f} {student_value:18.8f} {teacher_value:18.8f}")
    
print("PASS: all tracked teacher weights match the EMA formula.")

model.0.conv.weight
 Index  Both before SGD  Student after SGD  Teacher after EMA
   252      -0.38012695        -0.38348317        -0.38046259
   254      -0.47802734        -0.48137441        -0.47836205
   243       0.17541504         0.17216791         0.17509033
   245       0.37329102         0.37007001         0.37296891
   260      -0.64550781        -0.64868444        -0.64582551
   257      -1.15625000        -1.15941608        -1.15656662
   255      -0.95556641        -0.95873189        -0.95588297
   258      -0.56640625        -0.56955773        -0.56672138
PASS: all tracked teacher weights match the EMA formula.


In [14]:
# Evaluate both models on the same unaugmented validation images
# YOLO validator reads an isolated copy of the sample, leaving the source dataset alone.

scores = []

with TemporaryDirectory(prefix="aquatutor_poc_") as directory:

    sample_dir = Path(directory)
    image_dir = sample_dir / "images" / "val"
    label_dir = sample_dir / "labels" / "val"

    image_dir.mkdir(parents=True)
    label_dir.mkdir(parents=True)

    for index, (image, label) in enumerate(eval_pairs):
        copy2(image, image_dir / f"{index}{image.suffix.lower()}")
        copy2(label, label_dir / f"{index}.txt")

    data_yaml = sample_dir / "sample.yaml"

    data_yaml.write_text(yaml.safe_dump({
        "path": sample_dir.as_posix(),
        "train": "images/val",  # Required YAML field. Only val is evaluated here.
        "val": "images/val",
        "names": list(student.names.values()),
    }), encoding="utf-8")

    for name, model in (("Student", student), ("Teacher", teacher)):
        result = model.val(
            data=str(data_yaml), split="val", device=DEVICE, imgsz=IMAGE_SIZE,
            batch=min(4, X), workers=0, plots=False, verbose=False,
            project=str(sample_dir / "runs"), name=name,
        ).box

        scores.append((name, result.mp, result.mr, result.map50, result.map))

# print metrics
print(f"Detection metrics on {len(eval_pairs)} validation images")
print(f"{'Model':<10} {'Precision':>10} {'Recall':>8} {'mAP50':>8} {'mAP50–95':>10}")
for name, precision, recall, map50, map50_95 in scores:
    print(f"{name:<10} {precision:10.3f} {recall:8.3f} {map50:8.3f} {map50_95:10.3f}")

Ultralytics 8.4.108  Python-3.13.14 torch-2.10.0+cpu CPU (AMD Ryzen 7 5825U with Radeon Graphics)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 13,666 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 444.783.6 MB/s, size: 53.0 KB)
val: Scanning C:\Users\John\AppData\Local\Temp\aquatutor_poc_1_t3_jky\labels\val... 8 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 8/8 1.8Kit/s 0.0s
val: New cache created: C:\Users\John\AppData\Local\Temp\aquatutor_poc_1_t3_jky\labels\val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 7.3it/s 0.3s0.7s
                   all          8         11      0.902      0.996      0.995      0.741
Speed: 0.4ms preprocess, 16.4ms inference, 0.0ms loss, 0.7ms postprocess per image
Ultralytics 8.4.108  Python-3.13.14 torch-2.10.0+cpu CPU (AMD Ryzen 7 5825U with Radeon Graphics)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs


The weight table is the main PoC result. Similar or identical detection metrics are possible after one small update. To repeat the experiment, rerun from Step 1 so both models reload the original checkpoint.